## Google Colab (extension / colab.research.google.com)

1. **Runtime → Change runtime type → GPU** (T4+).
2. Đặt repo **`Pipeline`** dưới **`/content/Pipeline`** (clone từ Git hoặc upload / Drive rồi chỉnh `REPO_ROOT` ở cell đường dẫn).
3. Cell code dưới cài **PyTorch CUDA** + **mamba-ssm** (Linux + GPU). Trên **Windows** dùng Colab cloud hoặc WSL2 — không chạy Mamba build native trên Windows.


In [1]:
# **Mặc định: Google Colab** (Runtime → GPU; repo `/content/Pipeline`). VS Code Colab extension: cùng flow.
# Local Windows: cache ổ D: + dọn torch lỗi; Mamba không build trên Windows native.
# Thứ tự: numpy → torch (CUDA, `torch==…+cu…`) → causal-conv1d → mamba-ssm. Chỉ `torch` (không torchvision/torchaudio).

import os
import re
import shutil
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        pass
    # VS Code Colab extension / notebook VM: không luôn có package `google.colab`
    return os.path.isfile("/etc/colab-release") or (
        sys.platform == "linux" and os.path.isdir("/content")
    )


IN_COLAB = _in_colab()


def _prefer_d_drive_for_pip_temp_cache() -> None:
    if sys.platform != "win32":
        return
    if not os.path.isdir("D:\\"):
        print("[setup] không có D:\\ — pip dùng TEMP/cache mặc định", flush=True)
        return
    root = os.environ.get("PIPELINE_D_ROOT", r"D:\RobotDog_Pipeline_cache")
    os.makedirs(root, exist_ok=True)
    pip_cache = os.path.join(root, "pip-cache")
    tmp = os.path.join(root, "tmp")
    os.makedirs(pip_cache, exist_ok=True)
    os.makedirs(tmp, exist_ok=True)
    d_free = shutil.disk_usage("D:\\").free
    if d_free < 2 * (1024**3):
        raise RuntimeError(
            f"D:\\ chỉ còn ~{d_free // (1024**2)} MiB trống; cần ít nhất ~2 GiB trên D: cho pip cache/temp khi tải PyTorch."
        )
    os.environ["PIP_CACHE_DIR"] = pip_cache
    os.environ["TEMP"] = tmp
    os.environ["TMP"] = tmp
    print(f"[setup] PIP_CACHE_DIR + TEMP/TMP → {root}", flush=True)


_prefer_d_drive_for_pip_temp_cache()

print(
    f"[setup] {'colab' if IN_COLAB else 'local'} | cuda_install rev=9 | +cu pin; win: D:+nuke",
    flush=True,
)

_TORCH_FALLBACK_CU = {
    "cu124": "2.6.0+cu124",
    "cu121": "2.5.1+cu121",
    "cu118": "2.7.1+cu118",
}


def _pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


def _newest_torch_cu_version_on_index(index_url: str, tag: str) -> str:
    q = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "index",
            "versions",
            "torch",
            "--index-url",
            index_url,
        ],
        capture_output=True,
        text=True,
    )
    if q.returncode == 0:
        m = re.search(r"Available versions:\s*([0-9][^,\s]*)", q.stdout or "")
        if m:
            return m.group(1).strip()
    return _TORCH_FALLBACK_CU[tag]


def _pip_install_torch_cuda() -> None:
    import stat
    import site as _site
    from pathlib import Path as _Path

    def _rmtree_maybe_locked(path: _Path) -> None:
        def _onerr(fn, p, _exc):
            try:
                os.chmod(p, stat.S_IWRITE)
                fn(p)
            except OSError:
                pass

        if path.is_dir():
            shutil.rmtree(path, onerror=_onerr)

    if sys.platform == "win32":
        _drv = os.path.splitdrive(sys.executable)[0] + "\\"
        _free = shutil.disk_usage(_drv).free
        if _free < 3 * (1024**3):
            raise RuntimeError(
                f"Ổ {_drv} chỉ còn ~{_free // (1024**2)} MiB trống; cần ít nhất ~3 GiB để cài PyTorch CUDA "
                "(wheel lớn + giải nén). Giải phóng dĩa rồi chạy lại cell."
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torch"],
            capture_output=True,
            text=True,
        )
        for _sp in _site.getsitepackages():
            _root = _Path(_sp)
            _td = _root / "torch"
            _rmtree_maybe_locked(_td)
            for _di in _root.glob("torch-*.dist-info"):
                _rmtree_maybe_locked(_di)
            if _td.is_dir():
                _dead = _root / f"_torch_broken_{os.getpid()}"
                try:
                    _td.rename(_dead)
                    _rmtree_maybe_locked(_dead)
                except OSError:
                    pass
        _left: list[str] = []
        for _sp in _site.getsitepackages():
            _tp = _Path(_sp) / "torch"
            if _tp.is_dir():
                _left.append(str(_tp))
        if _left:
            raise RuntimeError(
                "Không xóa hết site-packages/torch (DLL thường bị khóa → WinError 5 khi pip ghi đè). "
                "Đóng mọi Jupyter kernel / Python / VS Code đang dùng env này, chạy lại cell; nếu vẫn lỗi, khởi động lại Windows rồi chạy lại.\n"
                + "\n".join(_left)
            )
        cuda_tags = ("cu121", "cu124", "cu118")
    else:
        cuda_tags = ("cu124", "cu121", "cu118")
    extra = ("--extra-index-url", "https://pypi.org/simple")
    last_log = ""
    for tag in cuda_tags:
        idx = f"https://download.pytorch.org/whl/{tag}"
        cu_ver = _newest_torch_cu_version_on_index(idx, tag)
        pkg = f"torch=={cu_ver}"
        p = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pkg,
                "--index-url",
                idx,
                *extra,
            ],
            capture_output=True,
            text=True,
        )
        if p.returncode == 0:
            chk = subprocess.run(
                [sys.executable, "-m", "pip", "show", "torch"],
                capture_output=True,
                text=True,
            )
            meta = chk.stdout or ""
            if "+cu" not in meta:
                last_log = (
                    "pip install thành công nhưng metadata không phải wheel CUDA (+cu). "
                    "Có thể vẫn sót torch\\lib cũ — đóng mọi kernel Python rồi chạy lại cell.\n"
                    + meta[:2000]
                )
                continue
            print(f"[pip] torch OK ({pkg}, index {tag})")
            return
        last_log = (p.stderr or p.stdout or "").strip()
    raise RuntimeError(
        "Không cài được PyTorch (CUDA). Đã thử: "
        + ", ".join(cuda_tags)
        + ".\n--- pip stderr/stdout (rút gọn) ---\n"
        + (last_log[:4000] if last_log else "(empty)")
    )


_pip("packaging", "ninja", "wheel", "numpy")
_pip_install_torch_cuda()
_pip("scikit-learn", "tensorboard", "einops")

# #region agent log
import json
import os
import time
import platform
import site
from pathlib import Path

def _agent_debug_log_path() -> str:
    import tempfile

    name = "debug-0c28ac.log"
    bases: list[str] = []
    if os.path.isdir("/content/Pipeline"):
        bases.append("/content/Pipeline")
    bases.append(os.getcwd())
    bases.append(tempfile.gettempdir())
    for base in bases:
        try:
            if base and os.path.isdir(base):
                return os.path.abspath(os.path.join(base, name))
        except OSError:
            continue
    return os.path.abspath(os.path.join(tempfile.gettempdir(), name))


print("[debug] agent log:", _agent_debug_log_path())


def _agent_dbg(hypothesis_id: str, location: str, message: str, data: dict) -> None:
    payload = {
        "sessionId": "0c28ac",
        "runId": "post-fix",
        "hypothesisId": hypothesis_id,
        "location": location,
        "message": message,
        "data": data,
        "timestamp": int(time.time() * 1000),
    }
    line = json.dumps(payload, ensure_ascii=False)
    try:
        with open(_agent_debug_log_path(), "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except OSError:
        print("AGENT_LOG", line, flush=True)


_agent_dbg(
    "H1",
    "compare_trajactory_predict_module.ipynb:pre_import_torch",
    "interpreter_and_platform",
    {
        "platform": sys.platform,
        "machine": platform.machine(),
        "architecture": list(platform.architecture()),
        "python_executable": sys.executable,
        "pointer_bits": __import__("struct").calcsize("P") * 8,
    },
)

_dll_info: dict = {}
_cuda_leaf = "torch_cuda.dll" if sys.platform == "win32" else "libtorch_cuda.so"
for sp in site.getsitepackages():
    dll = Path(sp) / "torch" / "lib" / _cuda_leaf
    if dll.is_file():
        st = dll.stat()
        _dll_info = {
            "path": str(dll),
            "size_bytes": st.st_size,
            "parent_list_head": [p.name for p in dll.parent.iterdir()][:30],
        }
        break
else:
    _dll_info = {"path": None, "size_bytes": None}
_agent_dbg("H2", "compare_trajactory_predict_module.ipynb:pre_import_torch", "torch_cuda_dll_stat", _dll_info)

_ps = subprocess.run(
    [sys.executable, "-m", "pip", "show", "-f", "torch"],
    capture_output=True,
    text=True,
)
_agent_dbg(
    "H3",
    "compare_trajactory_predict_module.ipynb:pre_import_torch",
    "pip_show_torch_files_head",
    {
        "returncode": _ps.returncode,
        "stdout_head": (_ps.stdout or "")[:3500],
        "stderr_head": (_ps.stderr or "")[:500],
    },
)

_tv = None
for _line in (_ps.stdout or "").splitlines():
    if _line.startswith("Version:"):
        _tv = _line.split(":", 1)[1].strip()
        break
_agent_dbg(
    "H5",
    "compare_trajactory_predict_module.ipynb:pre_import_torch",
    "torch_version_is_cuda_wheel",
    {"torch_version": _tv, "has_plus_cu": bool(_tv and "+cu" in _tv)},
)

_agent_dbg(
    "H4",
    "compare_trajactory_predict_module.ipynb:pre_import_torch",
    "msvc_and_cuda_env",
    {
        "CUDA_PATH": os.environ.get("CUDA_PATH"),
        "CUDA_HOME": os.environ.get("CUDA_HOME"),
        "path_has_cuda_bin": "cuda" in (os.environ.get("PATH", "").lower()),
        "system32_vcruntime140_1": (
            os.path.isfile(
                os.path.join(os.environ.get("SystemRoot", "C:\\Windows"), "System32", "vcruntime140_1.dll")
            )
            if sys.platform == "win32"
            else None
        ),
    },
)
# #endregion agent log

import torch

assert torch.cuda.is_available(), (
    "Cần GPU + PyTorch CUDA. Colab: Runtime → GPU. "
    "Linux: driver NVIDIA + torch CUDA."
)

# mamba-ssm / causal-conv1d: PyPI chủ yếu phát hành sdist; upstream ghi Linux + CUDA (Unix).
# Trên Windows native, pip gần như luôn cần biên dịch CUDA (nvcc) và hay lỗi — không hỗ trợ ở đây.
if sys.platform == "win32":
    raise RuntimeError(
        "Notebook (nhánh Mamba thật) không hỗ trợ cài mamba-ssm trên **Windows native**: "
        "PyPI không có wheel ổn định; causal-conv1d phải build CUDA.\n\n"
        "Cách làm đúng: **Google Colab (GPU)** hoặc **WSL2 Ubuntu** + PyTorch CUDA, rồi chạy lại từ đầu.\n"
        "Tham khảo: https://pypi.org/project/mamba-ssm/ (Operating System :: Unix)."
    )

_EXTRA = ("--extra-index-url", "https://pypi.org/simple")


def _pip_mamba_with_causal() -> None:
    """Cài mamba-ssm kèm causal-conv1d (extra chính thức). Thử --no-build-isolation nếu cần."""
    attempts = [
        [sys.executable, "-m", "pip", "install", "-q", "mamba-ssm[causal-conv1d]", *_EXTRA],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "mamba-ssm[causal-conv1d]",
            "--no-build-isolation",
            *_EXTRA,
        ],
    ]
    last = ""
    for cmd in attempts:
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode == 0:
            print("[pip] mamba-ssm[causal-conv1d] OK")
            return
        last = (p.stderr or p.stdout or "").strip()
    raise RuntimeError(
        "Không cài được mamba-ssm[causal-conv1d]. Đã thử pip thường và --no-build-isolation.\n"
        "Kiểm tra: Linux, CUDA khớp bản torch, đủ dung lượng build.\n\n"
        f"--- pip (rút gọn) ---\n{last[:5000]}"
    )


_pip_mamba_with_causal()

import mamba_ssm  # noqa: F401

print(
    "OK torch:",
    torch.__version__,
    "| cuda:",
    torch.version.cuda,
    "|",
    torch.cuda.get_device_name(0),
)


[setup] colab | cuda_install rev=9 | +cu pin; win: D:+nuke
[pip] torch OK (torch==2.6.0+cu124, index cu124)
[debug] agent log: /content/debug-0c28ac.log
[pip] mamba-ssm[causal-conv1d] OK


ImportError: /usr/local/lib/python3.12/dist-packages/selective_scan_cuda.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN3c107WarningC1ESt7variantIJNS0_11UserWarningENS0_18DeprecationWarningEEERKNS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEEb

In [5]:
# 1. Gỡ cài đặt các bản cũ nếu có để tránh xung đột
!pip uninstall -y torch torchvision torchaudio

# 2. Cài đặt Torch bản chuẩn cho CUDA 12.1 (Tương thích tốt nhất với Colab hiện tại)
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

# 3. Cài đặt các thư viện bổ trợ build
!pip install ninja packaging einops

# 4. Cài đặt bản mamba và conv1d từ file build sẵn (Sẽ không bị lỗi "Building wheel")
!pip install causal-conv1d==1.2.0.post2 --no-build-isolation
!pip install mamba-ssm==2.1.0 --no-build-isolation

# 5. Kiểm tra lại xem đã nhận GPU chưa
import torch
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 836.5 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 60.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.7 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 68.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 40.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 92.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━

## Nhiệm vụ notebook này

Luồng **end-to-end**: nhận data Stage A → train từng backbone thời gian → **so sánh hai nhóm metric**:

| Nhóm | Mục tiêu | Metric validation (in sau khi train) |
|------|-----------|--------------------------------------|
| **Risk** | Xác suất va chạm 0.5s / 1s / 2s | AP@0.5s, AP@1s, AP@2s, AUC@1s |
| **Quỹ đạo** | Dự đoán **H bước** tương lai `(x, y, yaw)` trong world (từ cùng `h_T`) | RMSE toàn cục, **ADE_xy / FDE_xy** (m), RMSE yaw (rad) |

1. **Data:** `data/stage_a_experiment/` (`index.jsonl` + `*.npz`; `ego_state` + nhãn `risk_*`).
2. **Train:** **Mamba (mamba-ssm thật, không GRU-giả-Mamba)** / GRU / LSTM / Transformer — PointPillars **frozen**; train reducer + temporal + **RiskHead + TrajectoryHead**; loss = focal BCE (risk) + trọng số × SmoothL1 (quỹ đạo).
3. **Output:** hai bảng in ra console + `summary.json` + TensorBoard + **file trọng số `.pt` theo từng backbone** (mặc định trong `runs/stage_a_compare/weights/`): `{tên}_stage_a_compare_<timestamp>.pt` và `{tên}_stage_a_compare.pt` (bản *latest* ghi đè mỗi lần train lại backbone đó).

**Chuẩn bị:** `python run_datagen_preset.py experiment`; checkpoint `PointPillars_module/pretrained/epoch_160.pt` hoặc `.pth`.

**Môi trường:** nhánh Mamba cần **Linux + CUDA** (Colab GPU hoặc WSL2 Ubuntu). **Windows native** không được hỗ trợ trong cell cài đặt (PyPI không có flow ổn định).


In [12]:
%ls

bin@      dev/     lib@     media/  python-apt/         sbin@  tools/
boot/     etc/     lib32@   mnt/    python-apt.tar.xz*  srv/   usr/
content/  home/    lib64@   opt/    root/               sys/   var/
datalab/  kaggle/  libx32@  proc/   run/                tmp/


In [6]:
# --- Bước 1–2: đường dẫn + kiểm tra data & checkpoint (chưa train) ---
# Colab / VS Code Colab extension: thường dùng `/content/Pipeline` sau `git clone`.
import os
import sys

REPO_ROOT = "/content/Pipeline"  # đổi nếu clone Drive: /content/drive/MyDrive/.../Pipeline
DATA_ROOT = os.path.join(REPO_ROOT, "data", "stage_a_experiment")
PRETRAINED_DIR = os.path.join(REPO_ROOT, "PointPillars_module", "pretrained")
CKPT = None
for _name in ("epoch_160.pt", "epoch_160.pth"):
    _p = os.path.join(PRETRAINED_DIR, _name)
    if os.path.isfile(_p):
        CKPT = _p
        break
LOG_ROOT = os.path.join(REPO_ROOT, "runs", "stage_a_compare")

# Huấn luyện — chỉnh tại đây (áp dụng cho mọi backbone)
EPOCHS = 3
BATCH_SIZE = 4
LR = 3e-4
MODELS = ("mamba", "gru", "lstm", "transformer")
SEED = 0
# H tương lai (20 Hz): 10 frame = 0.5 s — khớp mặc định RiskDataset / TrajectoryHead
TRAJ_HORIZON = 10
# Trọng số loss quỹ đạo so với risk (focal BCE)
TRAJ_LOSS_WEIGHT = 0.5
# None → lưu dưới LOG_ROOT/weights ; hoặc set chuỗi path tùy ý
WEIGHTS_DIR = None

for path in (REPO_ROOT, os.path.join(REPO_ROOT, "PointPillars_module"), os.path.join(REPO_ROOT, "create_dataset_module")):
    if path not in sys.path:
        sys.path.insert(0, path)
os.chdir(REPO_ROOT)

assert os.path.isdir(DATA_ROOT), f"Thiếu data: {DATA_ROOT} — chạy `python run_datagen_preset.py experiment` rồi upload."
assert os.path.isfile(os.path.join(DATA_ROOT, "index.jsonl")), f"Thiếu index.jsonl trong {DATA_ROOT}"
assert CKPT is not None, (
    f"Thiếu checkpoint trong {PRETRAINED_DIR}. "
    f"Cần epoch_160.pt hoặc epoch_160.pth."
)

print("OK — data:", DATA_ROOT)
print("OK — ckpt:", CKPT)
print("OK — sẽ train + so sánh:", MODELS)


FileNotFoundError: [Errno 2] No such file or directory: '/content/Pipeline'

In [ ]:
# --- Bước 3: train từng mô hình + in bảng so sánh (cùng một lần gọi) ---
from train_stage_a_compare import run_experiment

results = run_experiment(
    data_root=DATA_ROOT,
    ckpt_path=CKPT,
    models=MODELS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    log_root=LOG_ROOT,
    device=None,
    seed=SEED,
    traj_horizon=TRAJ_HORIZON,
    traj_loss_weight=TRAJ_LOSS_WEIGHT,
    weights_dir=WEIGHTS_DIR,
    save_weights=True,
    mamba_backend="mamba",  # không "auto": cấm GRU thay cho tên gọi "mamba"
)

**Sau train:** TensorBoard `%tensorboard --logdir runs/stage_a_compare`. **`summary.json`**: metrics + `checkpoint_pt` / `checkpoint_pt_latest`. Thư mục **`weights/`**: file `.pt` đầy đủ `model_state_dict` (CPU) + metadata (`backbone`, `pp_ckpt_path`, `traj_horizon`, …).

**Graph:** `pts_seq` → PointPillars (frozen) → reducer → temporal → **`h_T`** → **RiskHead** (3 logits) + **TrajectoryHead** (`H×3`).

Cell dưới in lại `summary.json` (risk + quỹ đạo + đường dẫn `.pt`).


In [ ]:
# --- Bước 4 (tuỳ chọn): đọc lại summary.json — hai bảng ---
import json
from pathlib import Path

summary_path = Path(LOG_ROOT) / "summary.json"
if not summary_path.is_file():
    print("Chưa có summary.json — chạy cell train trước.")
else:
    with summary_path.open(encoding="utf-8") as f:
        summary = json.load(f)
    print("File:", summary_path)
    print("\n[Risk]")
    print(f"{'model':<12} {'AP@0.5s':>8} {'AP@1s':>8} {'AP@2s':>8} {'AUC@1s':>8}")
    for name, m in summary.items():
        print(
            f"{name:<12} {m.get('ap_risk_05s', 'nan'):>8} {m.get('ap_risk_1s', 'nan'):>8} "
            f"{m.get('ap_risk_2s', 'nan'):>8} {m.get('auc_risk_1s', 'nan'):>8}"
        )
    print("\n[Trajectory H={} steps]".format(TRAJ_HORIZON))
    print(f"{'model':<12} {'RMSE_all':>10} {'ADE_xy':>10} {'FDE_xy':>10} {'RMSE_yaw':>10}")
    for name, m in summary.items():
        print(
            f"{name:<12} {m.get('traj_rmse_all', 'nan'):>10} {m.get('traj_ade_xy_m', 'nan'):>10} "
            f"{m.get('traj_fde_xy_m', 'nan'):>10} {m.get('traj_rmse_yaw_rad', 'nan'):>10}"
        )
    print("\n[Checkpoints .pt]")
    for name, m in summary.items():
        print(f"  {name}: {m.get('checkpoint_pt', '—')}")
        print(f"         (latest) {m.get('checkpoint_pt_latest', '—')}")